In [21]:
"""
====================================================
ERA5 Visualization Script
====================================================
"""

'\n====================================================\nERA5 Visualization Script\n====================================================\n'

In [22]:
#######################
#DIRECTORIES

In [23]:
# #SETTING UP DIRECTORIES
# mainDirectory = '/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/'
# workingDirectory="/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DataAnalysis/InputData_DataAnalysis/"
# print(workingDirectory)
# outputDirectory=workingDirectory+"OUTPUT/"
# dataDirectory=mainDirectory+"DownloadData/DATA/ERA5_Data/"

In [24]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
outputDirectory=mainDirectory+"../OUTPUT/DataAnalysis/InputData_DataAnalysis/"
import os; os.makedirs(outputDirectory, exist_ok=True)
dataDirectory=mainDirectory+"../DATA/ERA5_Data/"

In [25]:
#######################
#LIBRARIES, FUNCTIONS, and CLASSES

In [26]:
#IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Libraries/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [27]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [28]:
#IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/Classes/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Classes_InputData_DataAnalysis",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [29]:
###########################
#FUNCTIONS

In [30]:
#MAKE DATE FOLDER (for output) FUNCTION
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    #adding date to output folder
    subdir = os.path.join(outputDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder

def FixVariableName(variable,var_data):
    if variable == 'divergence':
        variable = 'convergence'
    return variable,var_data

def GetLoadDirectory(dataDirectory,date_folder):
    loadDirectorys = [
        os.path.join(dataDirectory, date_folder, f"{var}_ERA5_{date_folder}.nc")
        for var in variables.keys()
    ]
    return loadDirectorys

In [31]:
#RUN CALCULATIONS and PLOTTING
def RunCalculations(numerics, var_data, units, variable, calculation, mult_factor):
    calculation_results = {}

    arr   = var_data
    units = units
    if mult_factor != "NaN":
        arr *= mult_factor

    tz, _ = Ultimate_AreaAverage(var_data, dims=('t','z','y','x'), dim_names=('t','z'), mode='keep')
    t, _  = Ultimate_AreaAverage(tz,  dims=('t','z'),        dim_names=('t',),   mode='keep')

    three_hours = 3 * numerics.hour_index
    tz_3h   = calculation.block_vertical_profiles_2D(tz,  block=three_hours)
    tzyx_3h = calculation.block_vertical_profiles_4D(arr, block=three_hours)

    calculation_results[variable] = {
        "units": units,
        "tz": tz,          # (t,z)
        "t": t,            # (t,)
        "tz_3h": tz_3h,    # (nblocks, z)
        "tzyx_3h": tzyx_3h # (nblocks, z, y, x)
    }

    return calculation_results

def RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, colormap, vline, data_lim):
    for name, result in calculation_results.items():
        common_args = {
            "var_name": name,
            "var_units": result["units"],
            "date_string": date_string,
            "date_folder":  date_folder,
            "outputFile": outputFile,
            "numerics": numerics,
            "data_lim": data_lim,
            "UTC_offset": UTC_offset,
        }

        plotting.TZContourPlot(var_data=result['tz'], colormap=colormap, **common_args)
        plotting.TimeSeries(var_data=result['t'], **common_args)
        plotting.MultiAverage_VerticalProfiles(var_data=result['tz_3h'], vline=vline, **common_args)
        plotting.MultiAverage_HorizontalFields(var_data=result['tzyx_3h'], plev=1000, colormap=colormap, **common_args)

#RUNNING CALCULATIONS FUNCTION
def RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting):
    for count, (loadDirectory, (variable, components)) in enumerate(tqdm(zip(loadDirectorys, variables.items()), total=len(loadDirectorys), desc="Running Calculations"),start=1):
        units, colormap, vline, mult_factor = components["unit"], components["colormap"], components["vline"], components["mult_factor"]
        data_lim = components["data_lim"]
        
        #print
        print(f"Plotting {len(variables)} Variables",'\n')
        print(f"{count}. {variable} ({units}) → {loadDirectory}")
        
        #loading the variable
        ncFile=xr.open_dataset(loadDirectory)
        var_name = [v for v in list(ncFile.data_vars) if v not in ["number", "expver"]][0]
        # print('\n',var_name,'***')
        var_data=ncFile[var_name].data
        [variable,var_data] = FixVariableName(variable,var_data)
        numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
                           TIME=ncFile[var_name]['valid_time'].data,P=ncFile[var_name]['pressure_level'].data,LAT=ncFile[var_name]['latitude'].data,LON=ncFile[var_name]['longitude'].data)
        print(variable+":\n","\t(Nt, Np, Nlat, Nlon) = ",(numerics.Nt,numerics.Np,numerics.Nlat,numerics.Nlon),"\n")
    
        #making output filename
        outputFile = os.path.join(outputDirectory, date_folder, variable) #variable also can be var_name
        
        os.makedirs(outputFile, exist_ok=True)
    
        #doing calculations
        calculation_results=RunCalculations(numerics, var_data, "("+units+")", variable, calculation, mult_factor)
    
        #plotting
        RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, colormap, vline, data_lim)

In [32]:
###########################
#LOADING DATA

In [33]:
#load in ERA5 data
variables = {
    "u_component_of_wind": {"unit": r"$m\ s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN"},
    "v_component_of_wind": {"unit": r"$m\ s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN"},
    "vertical_velocity": {"unit": r"$Pa\ s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN"},
    "divergence": {"unit": r"$s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": -1, "data_lim": "NaN"}, #mult_factor: converts to convergence
    "vorticity": {"unit": r"$s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN"},
    "temperature": {"unit": r"$K$", "colormap": "coolwarm", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN"},
    "specific_humidity": {"unit": r"$g\ kg^{-1}$", "colormap": "YlGnBu", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN"},
    "specific_cloud_liquid_water_content": {"unit": r"$g\ kg^{-1}$", "colormap": "Blues", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN"},
    "specific_cloud_ice_water_content": {"unit": r"$g\ kg^{-1}$", "colormap": "Purples", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN"},
    "specific_rain_water_content": {"unit": r"$g\ kg^{-1}$", "colormap": "GnBu", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN"},
    "relative_humidity": {"unit": r"$\%$", "colormap": "BrBG", "vline": 0, "mult_factor": "NaN", "data_lim": (0,100)},
    "cloud_cover": {"unit": r"$1$", "colormap": "Greys", "vline": 0, "mult_factor": "NaN", "data_lim": (0,1)},
    "geopotential": {"unit": r"$m^{2}\ s^{-2}$", "colormap": "viridis", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN"}
}

In [34]:
###########################
#RUNNING

In [35]:
#TRACER DATA
UTC_offset="-5"

In [36]:
###########################
#DATE ONE (BORING CASE)

#date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/13 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-08_-_06-10_2022/u_component_of_wind_ERA5_06-08_-_06-10_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:   8%|▊         | 1/13 [00:09<01:50,  9.21s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

2. v_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-08_-_06-10_2022/v_component_of_wind_ERA5_06-08_-_06-10_2022.nc
v_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:  15%|█▌        | 2/13 [00:18<01:39,  9.06s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  85%|████████▍ | 11/13 [01:40<00:18,  9.25s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

12. cloud_cover ($1$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-08_-_06-10_2022/cloud_cover_ERA5_06-08_-_06-10_2022.nc
cloud_cover:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:  92%|█████████▏| 12/13 [01:49<00:09,  9.05s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

13. geopotential ($m^{2}\ s^{-2}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-08_-_06-10_2022/geopotential_ERA5_06-08_-_06-10_2022.nc
geopotential:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations: 100%|██████████| 13/13 [01:58<00:00,  9.10s/it]


In [37]:
###########################
#DATE TWO (RAINY CASE)

#date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/13 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/u_component_of_wind_ERA5_06-30_-_07-02_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:   8%|▊         | 1/13 [00:08<01:45,  8.80s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

2. v_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/v_component_of_wind_ERA5_06-30_-_07-02_2022.nc
v_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:  15%|█▌        | 2/13 [00:18<01:39,  9.04s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  85%|████████▍ | 11/13 [01:42<00:18,  9.12s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

12. cloud_cover ($1$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/cloud_cover_ERA5_06-30_-_07-02_2022.nc
cloud_cover:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:  92%|█████████▏| 12/13 [01:50<00:08,  8.92s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

13. geopotential ($m^{2}\ s^{-2}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/geopotential_ERA5_06-30_-_07-02_2022.nc
geopotential:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations: 100%|██████████| 13/13 [01:59<00:00,  9.22s/it]


In [38]:
###########################
#DATE THREE (INTERESTING CASE)

#date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/13 [00:00<?, ?it/s]

Plotting 13 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/08-11_-_08-13_2022/u_component_of_wind_ERA5_08-11_-_08-13_2022.nc


/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:   8%|▊         | 1/13 [00:08<01:45,  8.75s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  85%|████████▍ | 11/13 [01:40<00:18,  9.10s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

12. cloud_cover ($1$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/08-11_-_08-13_2022/cloud_cover_ERA5_08-11_-_08-13_2022.nc
cloud_cover:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations:  92%|█████████▏| 12/13 [01:48<00:08,  8.91s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_34073/1304748453.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 13 Variables 

13. geopotential ($m^{2}\ s^{-2}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/08-11_-_08-13_2022/geopotential_ERA5_08-11_-_08-13_2022.nc
geopotential:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations: 100%|██████████| 13/13 [01:57<00:00,  9.01s/it]


In [ ]:
######################################

In [ ]:
#PRECIP DATA (will need to make some changes to code to allow for differences in files between TRACER and PRECIP, or will conform the PRECIP nc metadata ***)
# UTC_offset=***

In [ ]:
###########################
#DATE ONE (BORING CASE)

In [ ]:
###########################
#DATE TWO (RAINY CASE)

In [ ]:
###########################
#DATE THREE (INTERESTING CASE)

In [ ]:
######################################

In [ ]:
#*#* Some Possible Future Improvements #*#*
#1. All Plots: x- and y-ticks somewhat incomplete